# prepare_datasets.ipynb

Pipeline canónico de preparación de datos — Seguro de Vida Individual (CNSF 2021–2024).

Fusiona y ordena las transformaciones de `data_cleaning.ipynb` y `eda_and_features.ipynb`.
Al finalizar escribe tres archivos Parquet listos para análisis y modelado.

**Flujo de transformaciones:**

| Paso | Operación | Origen |
|------|-----------|--------|
| 1 | Carga raw con `dtype=str` | `data_cleaning` |
| 2 | Eliminar filas duplicadas de encabezado (método 1: `isin`) | `data_cleaning` |
| 3 | Promover fila 0 como encabezado real | `data_cleaning` |
| 4 | Normalizar nombres de columna (strip, upper, snake_case, ANIO) | `eda_and_features` |
| 5 | Eliminar filas sentinel embebidas (método 2: centinelas explícitos) | `eda_and_features` |
| 6 | Casteo de columnas numéricas con `errors='coerce'` | `eda_and_features` |
| 7 | Diagnóstico: nulos, cardinalidad, distribución por año | `eda_and_features` |
| 8 | Columnas compartidas — candidatas a clave de join | `eda_and_features` |
| 9 | Exportar a Parquet | nuevo |


---
## 0. Imports y rutas

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path("../data/processed")
OUTPUT    = Path("../data/prepared")   # destino final — Parquet
OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Leyendo desde : {PROCESSED.resolve()}")
print(f"Escribiendo en: {OUTPUT.resolve()}")

Leyendo desde : /home/gabriel/research/ML-for-Insurance/data/processed
Escribiendo en: /home/gabriel/research/ML-for-Insurance/data/prepared


---
## 1. Carga raw

Todo como `str` para evitar que pandas infiera tipos erróneos sobre datos CNSF
que mezclan años, ceros y valores con formato de texto.

In [2]:
emision    = pd.read_csv(PROCESSED / "emision_total.csv",    dtype=str)
siniestros = pd.read_csv(PROCESSED / "siniestros_total.csv", dtype=str)
comisiones = pd.read_csv(PROCESSED / "comisiones_total.csv", dtype=str)

frames = {"emision": emision, "siniestros": siniestros, "comisiones": comisiones}

for name, df in frames.items():
    print(f"✓ {name:<12} {df.shape}")

✓ emision      (3508593, 12)
✓ siniestros   (297505, 13)
✓ comisiones   (827907, 16)


---
## 2. Eliminar filas que repiten encabezados (método isin)

> **Origen:** `data_cleaning.ipynb`  
> Los CSVs concatenados por año incluyen filas de encabezado intercaladas.
> Este paso elimina cualquier fila cuyo valor coincida con el nombre de alguna columna.

In [3]:
for name, df in frames.items():
    header_cols = df.columns.tolist()
    mascara     = df.isin(header_cols).any(axis=1)
    frames[name] = df[~mascara].reset_index(drop=True)
    print(f"{name}: {len(df):,} → {len(frames[name]):,} filas  ({mascara.sum():,} eliminadas)")

emision: 3,508,593 → 3,508,593 filas  (0 eliminadas)
siniestros: 297,505 → 297,505 filas  (0 eliminadas)
comisiones: 827,907 → 827,907 filas  (0 eliminadas)


---
## 3. Promover fila 0 como encabezado real

> **Origen:** `data_cleaning.ipynb`  
> Los CSVs tienen la fila 0 con los encabezados reales en lugar de los que pandas infirió.

In [4]:
for name, df in frames.items():
    nuevos_headers = df.iloc[0].tolist()
    frames[name]   = df.iloc[1:].copy()
    frames[name].columns = nuevos_headers
    frames[name] = frames[name].reset_index(drop=True)
    print(f"{name}: columnas → {nuevos_headers}")

emision: columnas → ['2021', 'EDAD', 'COBERTURA', 'PLAN DE LA POLIZA', 'MODALIDAD DE LA POLIZA', 'MONEDA', 'ENTIDAD ', 'SEXO', 'FORMA DE VENTA', 'NUMERO DE ASEGURADOS', 'PRIMA EMITIDA', 'SUMA ASEGURADA']
siniestros: columnas → ['2021', 'EDAD', 'COBERTURA', 'ENTIDAD', 'CAUSA DEL SINIESTRO', 'PLAN DE LA POLIZA', 'MODALIDAD DE LA POLIZA', 'SEXO', 'NUMERO DE SINIESTROS', 'MONTO RECLAMADO', 'VENCIMIENTOS', 'MONTO PAGADO', 'MONTO DE REASEGURO']
comisiones: columnas → ['2021', 'EDAD', 'PLAN DE LA POLIZA', 'MODALIDAD DE LA POLIZA', 'MONEDA', 'ENTIDAD ', 'SEXO', 'FORMA DE VENTA', 'TIPO DIVIDENDO', 'NUMERO DE ASEGURADOS', 'PRIMA CEDIDA', 'COMISIONES DIRECTAS', 'FONDO DE INVERSIÓN', 'FONDO DE ADMINISTRACION', 'MONTO DE DIVIDENDOS', 'MONTO DE RESCATE']


---
## 4. Normalizar nombres de columna

> **Origen:** `eda_and_features.ipynb`  
> Strip de espacios, uppercase, espacios internos → `_`, y renombrar la primera
> columna a `ANIO` si quedó con el valor literal del año (e.g. `'2021'`).

In [5]:
def normalize_columns(df):
    """Strip, uppercase y snake_case en nombres de columna. Renombra col[0] a ANIO si aplica."""
    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
        .str.replace(' ', '_', regex=False)
    )
    first_col = df.columns[0]
    if first_col != 'ANIO':
        df = df.rename(columns={first_col: 'ANIO'})
    return df

for name in frames:
    frames[name] = normalize_columns(frames[name])

for name, df in frames.items():
    print(f"\n[{name}] {list(df.columns)}")


[emision] ['ANIO', 'EDAD', 'COBERTURA', 'PLAN_DE_LA_POLIZA', 'MODALIDAD_DE_LA_POLIZA', 'MONEDA', 'ENTIDAD', 'SEXO', 'FORMA_DE_VENTA', 'NUMERO_DE_ASEGURADOS', 'PRIMA_EMITIDA', 'SUMA_ASEGURADA']

[siniestros] ['ANIO', 'EDAD', 'COBERTURA', 'ENTIDAD', 'CAUSA_DEL_SINIESTRO', 'PLAN_DE_LA_POLIZA', 'MODALIDAD_DE_LA_POLIZA', 'SEXO', 'NUMERO_DE_SINIESTROS', 'MONTO_RECLAMADO', 'VENCIMIENTOS', 'MONTO_PAGADO', 'MONTO_DE_REASEGURO']

[comisiones] ['ANIO', 'EDAD', 'PLAN_DE_LA_POLIZA', 'MODALIDAD_DE_LA_POLIZA', 'MONEDA', 'ENTIDAD', 'SEXO', 'FORMA_DE_VENTA', 'TIPO_DIVIDENDO', 'NUMERO_DE_ASEGURADOS', 'PRIMA_CEDIDA', 'COMISIONES_DIRECTAS', 'FONDO_DE_INVERSIÓN', 'FONDO_DE_ADMINISTRACION', 'MONTO_DE_DIVIDENDOS', 'MONTO_DE_RESCATE']


---
## 5. Eliminar filas sentinel embebidas

> **Origen:** `eda_and_features.ipynb`  
> Segunda pasada más quirúrgica: busca filas donde columnas clave contienen
> literalmente su propio nombre (e.g. `EDAD == 'EDAD'`).

In [6]:
SENTINEL_COLS = ["EDAD", "SEXO", "ENTIDAD", "COBERTURA", "PLAN_DE_LA_POLIZA"]

def drop_header_rows(df, sentinel_cols):
    mask = pd.Series(False, index=df.index)
    for col in sentinel_cols:
        if col in df.columns:
            mask |= df[col].astype(str).str.strip().str.upper() == col.upper()
    dropped = mask.sum()
    return df[~mask].copy(), dropped

for name in frames:
    frames[name], n = drop_header_rows(frames[name], SENTINEL_COLS)
    print(f"[{name}] filas sentinel eliminadas: {n:,}  →  shape: {frames[name].shape}")

[emision] filas sentinel eliminadas: 3  →  shape: (3508589, 12)
[siniestros] filas sentinel eliminadas: 3  →  shape: (297501, 13)
[comisiones] filas sentinel eliminadas: 3  →  shape: (827903, 16)


---
## 6. Casteo de columnas numéricas

> **Origen:** `eda_and_features.ipynb`  
> Convertimos a `float64` con `errors='coerce'` — los strings no parseables
> quedan como `NaN` y se reportan en el diagnóstico del paso 7.

In [7]:
NUM_COLS = {
    "emision": [
        "PRIMA_EMITIDA",
        "SUMA_ASEGURADA",
        "NUMERO_DE_ASEGURADOS",
    ],
    "siniestros": [
        "NUMERO_DE_SINIESTROS",
        "MONTO_RECLAMADO",
        "MONTO_PAGADO",
        "VENCIMIENTOS",
        "MONTO_DE_REASEGURO",
    ],
    "comisiones": [
        "NUMERO_DE_ASEGURADOS",
        "PRIMA_CEDIDA",
        "COMISIONES_DIRECTAS",
        "FONDO_DE_INVERSIÓN",
        "FONDO_DE_ADMINISTRACION",
        "MONTO_DE_DIVIDENDOS",
        "MONTO_DE_RESCATE",
    ],
}

def cast_numerics(df, num_cols):
    """Castea a float las columnas numéricas que existan en df."""
    existing = [c for c in num_cols if c in df.columns]
    missing  = [c for c in num_cols if c not in df.columns]
    if missing:
        print(f"  ⚠️  Columnas no encontradas: {missing}")
    for c in existing:
        df[c] = pd.to_numeric(
            df[c].str.strip().str.replace(',', '', regex=False),
            errors='coerce'
        )
    return df

for name in frames:
    print(f"\n[{name}]")
    frames[name] = cast_numerics(frames[name], NUM_COLS.get(name, []))

# Reasignar referencias individuales
emision_f    = frames["emision"]
siniestros_f = frames["siniestros"]
comisiones_f = frames["comisiones"]

print("\n✓ Casteo completado")


[emision]

[siniestros]

[comisiones]

✓ Casteo completado


---
## 7. Diagnóstico post-limpieza

> **Origen:** `eda_and_features.ipynb` (secciones 3–6)  
> Ejecutar para verificar que el pipeline produjo datos coherentes.
> No modifica ningún DataFrame.

In [8]:
# ── 7a. Estadísticas descriptivas — numéricas ────────────────────────────────
PERCENTILES = [0.25, 0.50, 0.75, 0.95]

for name, df in frames.items():
    num_df = df.select_dtypes(include='number')
    if num_df.empty:
        print(f"\n[{name}] — sin columnas numéricas")
        continue
    desc = num_df.describe(percentiles=PERCENTILES).T
    desc.columns = [c.upper() for c in desc.columns]
    print(f"\n{'='*70}")
    print(f"  {name.upper()} — estadísticas numéricas")
    print(f"{'='*70}")
    print(desc.to_string(float_format=lambda x: f"{x:,.2f}"))


  EMISION — estadísticas numéricas
                            COUNT          MEAN            STD            MIN       25%        50%          75%           95%               MAX
NUMERO_DE_ASEGURADOS 3,508,589.00         60.00         518.18           1.00      2.00       5.00        23.00        257.00        130,071.00
PRIMA_EMITIDA        3,508,589.00    248,091.69   3,198,567.96 -14,894,612.05    448.32   4,394.01    37,568.77    635,092.05  2,448,270,922.42
SUMA_ASEGURADA       3,508,589.00 19,452,081.65 102,688,613.09           0.00 60,008.43 995,821.00 6,469,399.00 81,257,198.08 66,326,216,079.25

  SINIESTROS — estadísticas numéricas
                          COUNT       MEAN          STD             MIN    25%       50%        75%          95%            MAX
NUMERO_DE_SINIESTROS 297,501.00       2.40        10.64            1.00   1.00      1.00       2.00         7.00       3,304.00
MONTO_RECLAMADO      297,501.00 249,207.14 1,922,788.72  -61,776,356.79   0.01 34,000.00 200,

In [9]:
# ── 7b. Diagnóstico de nulos ─────────────────────────────────────────────────
def null_report(df, name):
    total      = len(df)
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / total * 100).round(2)
    report = pd.DataFrame({
        "nulos": null_counts,
        "pct_%": null_pct,
        "dtype": df.dtypes,
    })
    report = report[report["nulos"] > 0].sort_values("pct_%", ascending=False)
    print(f"\n{'='*55}")
    print(f"  {name.upper()} — columnas con nulos ({total:,} filas total)")
    print(f"{'='*55}")
    if report.empty:
        print("  ✅ Sin nulos detectados")
    else:
        print(report.to_string())

for name, df in frames.items():
    null_report(df, name)


  EMISION — columnas con nulos (3,508,589 filas total)
      nulos  pct_% dtype
EDAD     11    0.0   str

  SINIESTROS — columnas con nulos (297,501 filas total)
                   nulos  pct_% dtype
PLAN_DE_LA_POLIZA      1    0.0   str

  COMISIONES — columnas con nulos (827,903 filas total)
  ✅ Sin nulos detectados


In [10]:
# ── 7c. Cobertura temporal por año ───────────────────────────────────────────
print(f"{'TABLA':<15} {'ANIO':<8} {'FILAS':>10} {'PCT':>8}")
print("-" * 45)

for name, df in frames.items():
    if "ANIO" not in df.columns:
        print(f"  ⚠️  {name}: columna 'ANIO' no encontrada")
        continue
    vc    = df["ANIO"].value_counts().sort_index()
    total = len(df)
    for anio, cnt in vc.items():
        pct = cnt / total * 100
        print(f"{name:<15} {str(anio):<8} {cnt:>10,} {pct:>7.1f}%")
    print("-" * 45)

TABLA           ANIO          FILAS      PCT
---------------------------------------------
emision         2021        816,498    23.3%
emision         2022        879,533    25.1%
emision         2023        888,268    25.3%
emision         2024        924,290    26.3%
---------------------------------------------
siniestros      2021         81,107    27.3%
siniestros      2022         72,622    24.4%
siniestros      2023         71,235    23.9%
siniestros      2024         72,537    24.4%
---------------------------------------------
comisiones      2021        211,493    25.5%
comisiones      2022        204,954    24.8%
comisiones      2023        200,254    24.2%
comisiones      2024        211,202    25.5%
---------------------------------------------


---
## 8. Columnas compartidas — candidatas a clave compuesta de join

> **Origen:** `eda_and_features.ipynb` (sección 7)  
> Identifica las columnas presentes en las tres tablas: son las candidatas
> naturales para construir la clave compuesta de join sin foreign key explícito.

In [11]:
col_sets = {name: set(df.columns) for name, df in frames.items()}

shared_all   = col_sets["emision"] & col_sets["siniestros"] & col_sets["comisiones"]
shared_em_si = col_sets["emision"] & col_sets["siniestros"]
shared_em_co = col_sets["emision"] & col_sets["comisiones"]
shared_si_co = col_sets["siniestros"] & col_sets["comisiones"]

print("Columnas en las TRES tablas (candidatas a clave compuesta):")
print(sorted(shared_all))

print("\nColumnas emision ∩ siniestros (no en comisiones):")
print(sorted(shared_em_si - col_sets["comisiones"]))

print("\nColumnas emision ∩ comisiones (no en siniestros):")
print(sorted(shared_em_co - col_sets["siniestros"]))

print("\nColumnas exclusivas por tabla:")
for name, cs in col_sets.items():
    excl = cs - (shared_em_si | shared_em_co | shared_si_co)
    print(f"  {name}: {sorted(excl)}")

Columnas en las TRES tablas (candidatas a clave compuesta):
['ANIO', 'EDAD', 'ENTIDAD', 'MODALIDAD_DE_LA_POLIZA', 'PLAN_DE_LA_POLIZA', 'SEXO']

Columnas emision ∩ siniestros (no en comisiones):
['COBERTURA']

Columnas emision ∩ comisiones (no en siniestros):
['FORMA_DE_VENTA', 'MONEDA', 'NUMERO_DE_ASEGURADOS']

Columnas exclusivas por tabla:
  emision: ['PRIMA_EMITIDA', 'SUMA_ASEGURADA']
  siniestros: ['CAUSA_DEL_SINIESTRO', 'MONTO_DE_REASEGURO', 'MONTO_PAGADO', 'MONTO_RECLAMADO', 'NUMERO_DE_SINIESTROS', 'VENCIMIENTOS']
  comisiones: ['COMISIONES_DIRECTAS', 'FONDO_DE_ADMINISTRACION', 'FONDO_DE_INVERSIÓN', 'MONTO_DE_DIVIDENDOS', 'MONTO_DE_RESCATE', 'PRIMA_CEDIDA', 'TIPO_DIVIDENDO']


---
## 9. Exportar a Parquet

Parquet preserva dtypes (float64 vs object), comprime bien datos con muchos
valores repetidos y es ~10x más rápido de leer que CSV a estas escalas.

Los notebooks de análisis arrancan directamente desde aquí con un `pd.read_parquet(...)`.

In [12]:
for name, df in frames.items():
    path = OUTPUT / f"{name}.parquet"
    df.to_parquet(path, index=False, engine="pyarrow", compression="snappy")
    size_mb = path.stat().st_size / 1_048_576
    print(f"✓ {name:<12} → {path.name}  {df.shape}  ({size_mb:.1f} MB)")

print("\n✅ Pipeline completado. Datasets listos en:", OUTPUT.resolve())

✓ emision      → emision.parquet  (3508589, 12)  (54.6 MB)
✓ siniestros   → siniestros.parquet  (297501, 13)  (4.6 MB)
✓ comisiones   → comisiones.parquet  (827903, 16)  (12.9 MB)

✅ Pipeline completado. Datasets listos en: /home/gabriel/research/ML-for-Insurance/data/prepared


---
## ⚡ Quick start — sesiones posteriores

Si este notebook ya fue ejecutado, cualquier notebook de análisis puede partir
directamente desde aquí sin re-ejecutar el pipeline completo.

In [ ]:
# ── Cargar datasets preparados ───────────────────────────────────────────────
# import pandas as pd
# from pathlib import Path
#
# OUTPUT = Path("../data/prepared")
#
# emision_f    = pd.read_parquet(OUTPUT / "emision.parquet")
# siniestros_f = pd.read_parquet(OUTPUT / "siniestros.parquet")
# comisiones_f = pd.read_parquet(OUTPUT / "comisiones.parquet")
#
# frames = {"emision": emision_f, "siniestros": siniestros_f, "comisiones": comisiones_f}
#
# for name, df in frames.items():
#     print(f"✓ {name:<12} {df.shape}")